In [1]:
import sys
from pathlib import Path


def find_project_root() -> Path:
    current = Path.cwd().resolve()

    for directory in [current, *current.parents]:
        if (directory / "pyproject.toml").exists():
            return directory

    raise FileNotFoundError("Proje ana klasörü bulunamadı.")


PROJECT_ROOT = find_project_root()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

DATA_PATH = PROJECT_ROOT / "data" / "reference_material.json.gz"

print("Proje:", PROJECT_ROOT)
print("Veri:", DATA_PATH)

Proje: C:\Users\kerem\Desktop\capsstone
Veri: C:\Users\kerem\Desktop\capsstone\data\reference_material.json.gz


In [2]:
from src.rag import load_documents

all_documents = load_documents(DATA_PATH, deduplicate=False)
documents = load_documents(DATA_PATH, deduplicate=True)

print("Ham doküman sayısı:", len(all_documents))
print("Benzersiz doküman sayısı:", len(documents))
print("Silinen tekrar sayısı:", len(all_documents) - len(documents))

Ham doküman sayısı: 1000
Benzersiz doküman sayısı: 994
Silinen tekrar sayısı: 6


In [3]:
first_document = documents[0]

print("ID:", first_document.document_id)
print("Başlık:", first_document.title)
print("Kategori:", first_document.category)
print("Sorun kodu:", first_document.issue_code)
print("Tekrarlı ID'ler:", first_document.duplicate_ids)
print()
print(first_document.content)

ID: DOC000001
Başlık: Yazilim Sorunu
Kategori: Donanim
Sorun kodu: firmware_issue
Tekrarlı ID'ler: ()

# Yazilim Sorunu

**Kategori:** Donanim

## Sorun
Son yazilim guncellemesinden sonra modem duzgun calismiyor.

## Cozum Adimlari
1. Sebeke ayarlarini sifirlayin.
2. Cihazinizi kapatip yeniden baslatin.
3. Uygulamayi guncel surume yukseltin.
4. Fatura detaylarini hesabim sayfasindan inceleyin.
5. SIM karti cikarip yeniden takin.

## Notlar
Bu dokuman Donanim kategorisindeki 'firmware_issue' sorununun giderilmesi icin hazirlanmistir.


In [3]:
import os

from dotenv import load_dotenv
from openai import OpenAI

from src.rag import Embedder, FaissStore


load_dotenv(PROJECT_ROOT / ".env")

api_key = os.getenv("API_KEY")
base_url = os.getenv(
    "DEEPINFRA_BASE_URL",
    "https://api.deepinfra.com/v1/openai",
)
embedding_model = os.getenv(
    "EMBEDDING_MODEL",
    "BAAI/bge-m3",
)

if not api_key:
    raise ValueError(".env dosyasında API_KEY bulunamadı.")

client = OpenAI(
    api_key=api_key,
    base_url=base_url,
)

embedder = Embedder(
    client=client,
    model=embedding_model,
    batch_size=64,
)

print("Embedding modeli:", embedder.model)

Embedding modeli: BAAI/bge-m3


In [5]:
INDEX_PATH = PROJECT_ROOT / "data" / "rag_index"

store = FaissStore.build(
    documents=documents,
    embedder=embedder,
)

store.save(INDEX_PATH)

print("İndekslenen doküman:", store.index.ntotal)
print("İndeks konumu:", INDEX_PATH)

İndekslenen doküman: 994
İndeks konumu: C:\Users\kerem\Desktop\capsstone\data\rag_index


In [6]:
store = FaissStore.load(
    directory=INDEX_PATH,
    embedder=embedder,
)

print("İndeks yüklendi:", store.index.ntotal)

İndeks yüklendi: 994


**Deneme Amaçlı**

In [9]:
results = store.search(
    query="Modemim sürekli kendi kendine yeniden başlıyor.",
    top_k=5,
)

for rank, result in enumerate(results, start=1):
    print(
        f"{rank}. skor={result.score:.4f} | "
        f"id={result.document.document_id} | "
        f"başlık={result.document.title} | "
        f"kategori={result.document.category}"
    )

best_result = results[0]

print(best_result.document.content)
print("Sorun kodu:", best_result.document.issue_code)
print("Benzerlik:", best_result.score)

results = store.search(
    query="Modemim sürekli kendi kendine yeniden başlıyor.",
    top_k=1,
)

1. skor=0.7764 | id=DOC000597 | başlık=Modem Arizasi | kategori=Donanim
2. skor=0.7760 | id=DOC000863 | başlık=Modem Arizasi | kategori=Donanim
3. skor=0.7752 | id=DOC000349 | başlık=Modem Arizasi | kategori=Donanim
4. skor=0.7746 | id=DOC000450 | başlık=Modem Arizasi | kategori=Donanim
5. skor=0.7746 | id=DOC000361 | başlık=Modem Arizasi | kategori=Donanim
# Modem Arizasi

**Kategori:** Donanim

## Sorun
Modemim kendi kendine surekli yeniden basliyor.

## Cozum Adimlari
1. Sinyal seviyesini farkli bir konumda test edin.
2. Modeminizin kablolarini kontrol edin.
3. Sorun devam ederse musteri hizmetleri ile iletisime gecin.
4. Sebeke ayarlarini sifirlayin.

## Notlar
Bu dokuman Donanim kategorisindeki 'router_malfunction' sorununun giderilmesi icin hazirlanmistir.
Sorun kodu: router_malfunction
Benzerlik: 0.7764420509338379


In [15]:
import importlib
import sys

importlib.invalidate_caches()

for module_name in list(sys.modules):
    if module_name == "src.rag" or module_name.startswith("src.rag."):
        del sys.modules[module_name]

from src.rag import RAGChatbot

print("Import başarılı:", RAGChatbot)

Import başarılı: <class 'src.rag.chatbot.RAGChatbot'>


In [16]:
from src.rag import RAGChatbot

chat_model = os.getenv(
    "CHAT_MODEL",
    "Qwen/Qwen3.6-35B-A3B",
)

chatbot = RAGChatbot(
    client=client,
    store=store,
    chat_model=chat_model,
    top_k=3,
    min_score=0.35,
    score_margin=0.08,
)

response = chatbot.ask(
    "Modemim sürekli yeniden başlıyor, ne yapmalıyım?"
)

print(response.answer)

print("\nKaynaklar:")

for source in response.sources:
    print(
        source.document.document_id,
        source.document.title,
        source.document.issue_code,
        round(source.score, 4),
    )

Modeminizin sürekli yeniden başlaması sorunu için aşağıdaki adımları uygulayabilirsiniz:

1. Modeminizin kablolarını kontrol edin.
2. Cihazınızı kapatıp yeniden başlatın.
3. Sorun devam ederse müşteri hizmetleri ile iletişime geçin.
4. SIM kartını çıkarıp yeniden takın.
5. Sinyal seviyesini farklı bir konumda test edin.

Kaynaklar: DOC000361

Kaynaklar:
DOC000361 Modem Arizasi router_malfunction 0.7793


In [17]:
chat_model = os.getenv(
    "CHAT_MODEL",
    "Qwen/Qwen3.6-35B-A3B",
)

chatbot = RAGChatbot(
    client=client,
    store=store,
    chat_model=chat_model,
    top_k=3,
    min_score=0.35,
    score_margin=0.08,
)

In [ ]:
response = chatbot.ask(
    "Modemim sürekli yeniden başlıyor, ne yapmalıyım?"
)

print(response.answer)

print("\nGetirilen kaynaklar:")

for source in response.sources:
    print(
        f"- {source.document.document_id} | "
        f"{source.document.title} | "
        f"{source.document.issue_code} | "
        f"{source.score:.4f}"
    )

In [18]:
from src.rag import evaluate_retrieval

evaluation = evaluate_retrieval(
    store=store,
    top_k=3,
)

print("Test sayısı:", evaluation.total)
print(f"Hit@1: {evaluation.hit_at_1:.2%}")
print(f"Hit@3: {evaluation.hit_at_3:.2%}")
print(f"MRR: {evaluation.mean_reciprocal_rank:.4f}")
print("Başarısız testler:", evaluation.failures)

Test sayısı: 7
Hit@1: 85.71%
Hit@3: 85.71%
MRR: 0.8571
Başarısız testler: [{'question': 'Evde telefonum hiç çekmiyor.', 'expected': 'indoor_coverage', 'predicted': ['no_signal', 'no_signal', 'no_signal']}]
